# Gold dc_map -- data-center map (Lab 12 Part B)

Builds the `dc_map` gold table that powers the data-center map dashboard: one point per
data center, GREEN when it has a recent non-zero energy-consumption reading and RED when
it does not.

Data-center metadata comes from `neon.public.dc_dim` (Lakehouse Federation, see
`Lab10/databricks_federation_datacenters.sql`). It is first snapshotted into this
project's own bronze schema so the dashboard refresh does not depend on Neon being
reachable, then joined to the latest `consumption_hourly` reading per site.

Prod-only: dev runs on Databricks Free Edition, which has no `neon` foreign catalog.

In [0]:
# Configuration
from pyspark.sql import functions as F
dbutils.widgets.combobox("catalog", "dbr_dev", ["workspace", "dbr_dev"], "Unity Catalog")
dbutils.widgets.combobox("bronze_schema", "gabrielajaniszews786_bronze", ["bronze", "gabrielajaniszews786_bronze"], "Bronze schema")
dbutils.widgets.combobox("gold_schema", "gabrielajaniszews786_gold", ["gold", "gabrielajaniszews786_gold"], "Gold schema")
CATALOG       = dbutils.widgets.get("catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")
GOLD_SCHEMA   = dbutils.widgets.get("gold_schema")

DC_DIM_SNAPSHOT    = f"{CATALOG}.{BRONZE_SCHEMA}.dc_dim_snapshot"
CONSUMPTION_HOURLY = f"{CATALOG}.{GOLD_SCHEMA}.consumption_hourly"
DC_MAP             = f"{CATALOG}.{GOLD_SCHEMA}.dc_map"


In [0]:
# Bronze snapshot of the Neon dc_dim reference table. The dashboard reads this copy so a
# refresh never fails because Neon is unreachable; re-run this notebook to pick up changes.
spark.sql(f"""
CREATE OR REPLACE TABLE {DC_DIM_SNAPSHOT}
AS SELECT *, current_timestamp() AS snapshot_ts
FROM neon.public.dc_dim
""")


In [0]:
# Gold dc_map -- drives from the dc_dim snapshot (not from consumption_hourly) so every
# data center appears exactly once, including ones with no consumption at all -- those
# are the points that must render red on the map. Each center is enriched with its
# single latest consumption reading, if any.
spark.sql(f"""
CREATE OR REPLACE TABLE {DC_MAP} AS
SELECT
  dc.dc_id,
  dc.dc_name,
  dc.operator,
  dc.country_code,
  dc.bidding_zone,
  dc.city,
  dc.latitude,
  dc.longitude,
  dc.tech_area_m2,
  dc.it_power_mw,
  dc.power_density_kw_m2,
  dc.tier,
  dc.pue,
  dc.commissioned_year,
  dc.cooling_type,
  latest.date               AS latest_reading_date,
  latest.hour               AS latest_reading_hour,
  latest.avg_consumption_kwh,
  latest.avg_power_kw,
  latest.avg_pue             AS latest_avg_pue,
  latest.cost_per_hour,
  CASE
    WHEN latest.avg_consumption_kwh IS NOT NULL AND latest.avg_consumption_kwh > 0 THEN true
    ELSE false
  END AS is_energy_consumed
FROM {DC_DIM_SNAPSHOT} AS dc
LEFT JOIN (
  SELECT *
  FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY site_id ORDER BY date DESC, hour DESC) AS rn
    FROM {CONSUMPTION_HOURLY}
  )
  WHERE rn = 1
) AS latest
  ON dc.dc_id = latest.site_id
""")


In [0]:
# Sanity checks -- every data center must appear exactly once, and every point must be
# placeable on the map.
dc_count = spark.table(DC_DIM_SNAPSHOT).select("dc_id").distinct().count()
map_count = spark.table(DC_MAP).count()
assert dc_count == map_count, f"dc_map has {map_count} rows, expected {dc_count} (one per data center)"

null_coords = spark.table(DC_MAP).filter("latitude IS NULL OR longitude IS NULL").count()
assert null_coords == 0, f"{null_coords} dc_map rows have a null latitude/longitude"

print(f"dc_map OK: {map_count} data centers, all with coordinates")
